# Coringa Mangrove Classification — Landsat-8 2017

Comparing RF, XGBoost, and DeepLabV3+ on Landsat-8 OLI data.

**Config:**
- Sensor: Landsat-8 OLI (30m resolution)
- Date: Sep–Dec 2017
- Patch size: 64×64 (1920m × 1920m per patch)
- Features: 6 Landsat bands + 5 spectral indices = 11 channels
- Labels: ESA WorldCover 2021 (class 95 = mangroves)

In [ ]:
!pip install earthengine-api geemap scikit-learn xgboost segmentation_models_pytorch -q
!earthengine authenticate

In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

ee.Initialize(project='coringa-mangrove')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Constants

In [ ]:
CORINGA_BBOX = [82.13, 16.42, 82.23, 17.0]
REGION = ee.Geometry.Rectangle(CORINGA_BBOX)

# Landsat-8 date range (Sep-Dec 2017, matching paper)
START_DATE = '2017-09-01'
END_DATE = '2017-12-31'

# Landsat-8 band names
L8_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7']  # Blue, Green, Red, NIR, SWIR1, SWIR2

# Patch config (30m resolution, use smaller patches)
PATCH_SIZE = 64

# Training config
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-4

# Spectral indices
INDEX_BANDS = ['NDVI', 'NDMI', 'MNDWI', 'GCVI', 'SR']
ALL_FEATURES = L8_BANDS + INDEX_BANDS
N_CHANNELS = len(ALL_FEATURES)  # 11

MANGROVE_CLASS = 95

## 2. Load Landsat-8 & Compute Indices

In [ ]:
def load_landsat8():
    """Load and composite Landsat-8 OLI for the study period."""
    l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterBounds(REGION)
          .filterDate(START_DATE, END_DATE)
          .filter(ee.Filter.lt('CLOUD_COVER', 20))
          .median()
          .clip(REGION))
    # Apply scaling factors (Collection 2 Level 2)
    # Optical: scale = 0.0000275, offset = -0.2
    scaled = l8.multiply(0.0000275).add(-0.2)
    return scaled.select(L8_BANDS)

def compute_indices(image):
    """Compute spectral indices from Landsat-8 bands."""
    # Landsat-8: B5=NIR, B4=Red, B3=Green, B6=SWIR1, B7=SWIR2
    ndvi = image.normalizedDifference(['B5', 'B4']).rename('NDVI')
    ndmi = image.normalizedDifference(['B5', 'B7']).rename('NDMI')
    mndwi = image.normalizedDifference(['B3', 'B6']).rename('MNDWI')
    gcvi = image.expression('B5 / B3 - 1', {
        'B5': image.select('B5'), 'B3': image.select('B3')
    }).rename('GCVI')
    sr = image.expression('B5 / B4', {
        'B5': image.select('B5'), 'B4': image.select('B4')
    }).rename('SR')
    return image.addBands([ndvi, ndmi, mndwi, gcvi, sr])

def prepare_features():
    """Full pipeline: load L8, compute indices, return stacked image."""
    l8 = load_landsat8()
    with_indices = compute_indices(l8)
    return with_indices.select(ALL_FEATURES)

print('Loading Landsat-8 2017 imagery...')
features_image = prepare_features()
print('Feature bands:', features_image.bandNames().getInfo())

## 3. Load WorldCover Labels

In [ ]:
def load_worldcover():
    """Load ESA WorldCover 2021 and create binary mangrove mask."""
    wc = (ee.ImageCollection('ESA/WorldCover/v200/2021')
          .filterBounds(REGION)
          .first()
          .clip(REGION))
    mangrove_mask = wc.eq(MANGROVE_CLASS).rename('mangrove')
    return mangrove_mask

print('Loading WorldCover labels...')
labels_image = load_worldcover()
print('Labels loaded.')

## 4. Download to Numpy & Extract Patches

In [ ]:
# Combine features + labels into one image
combined = features_image.addBands(labels_image)

print('Downloading combined image (Landsat-8 30m)...')
combined_array = geemap.ee_to_numpy(combined, region=REGION, scale=30)
print(f'Combined shape: {combined_array.shape}')

# Split features and labels
features_array = combined_array[:, :, :N_CHANNELS]
labels_array = combined_array[:, :, N_CHANNELS:]

print(f'Features: {features_array.shape}, Labels: {labels_array.shape}')
print(f'Mangrove ratio: {(labels_array == 1).mean():.4f}')

In [ ]:
# Extract patches from numpy array
def extract_patches(features, labels, patch_size=64, n_patches=1000):
    h, w, c = features.shape
    X_list = []
    y_list = []

    np.random.seed(42)
    for i in range(n_patches):
        r = np.random.randint(0, h - patch_size)
        c_idx = np.random.randint(0, w - patch_size)

        patch_x = features[r:r+patch_size, c_idx:c_idx+patch_size]
        patch_y = labels[r:r+patch_size, c_idx:c_idx+patch_size]

        X_list.append(patch_x.transpose(2, 0, 1))  # (C, H, W)
        y_list.append(patch_y.transpose(2, 0, 1))  # (1, H, W)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)

    # Normalize per channel
    for c in range(X.shape[1]):
        mean = X[:, c].mean()
        std = X[:, c].std() + 1e-8
        X[:, c] = (X[:, c] - mean) / std

    return X, y

X, y = extract_patches(features_array, labels_array, PATCH_SIZE, n_patches=1000)
print(f'X: {X.shape}, y: {y.shape}')
print(f'Mangrove ratio: {y.mean():.4f}')

## 5. Dataset & DataLoader

In [ ]:
class MangroveDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Split: 70/15/15
n = len(X)
idx = np.random.permutation(n)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

train_loader = DataLoader(MangroveDataset(X[idx[:n_train]], y[idx[:n_train]]),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(MangroveDataset(X[idx[n_train:n_train+n_val]], y[idx[n_train:n_train+n_val]]),
                        batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(MangroveDataset(X[idx[n_train+n_val:]], y[idx[n_train+n_val:]]),
                         batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {n_train}, Val: {n_val}, Test: {n - n_train - n_val}')

## 6. RF & XGBoost (pixel-level)

In [ ]:
# Flatten patches to pixels
X_pixels = X.reshape(X.shape[0], X.shape[1], -1).transpose(0, 2, 1).reshape(-1, N_CHANNELS)
y_pixels = y.reshape(-1)

# Subsample 100k pixels for speed
sub_idx = np.random.choice(len(X_pixels), 100000, replace=False)
X_sub = X_pixels[sub_idx]
y_sub = y_pixels[sub_idx]

X_train_mL, X_test_mL, y_train_mL, y_test_mL = train_test_split(
    X_sub, y_sub, test_size=0.3, random_state=42, stratify=y_sub
)
print(f'Train: {len(X_train_mL)}, Test: {len(X_test_mL)}')

In [ ]:
# RF
rf = RandomForestClassifier(n_estimators=100, max_features=5, random_state=42, n_jobs=-1)
rf.fit(X_train_mL, y_train_mL)
y_pred_rf = rf.predict(X_test_mL)

def compute_iou(y_true, y_pred):
    intersection = (y_true.astype(bool) & y_pred.astype(bool)).sum()
    union = (y_true.astype(bool) | y_pred.astype(bool)).sum()
    return intersection / (union + 1e-7)

print(f'RF Accuracy: {accuracy_score(y_test_mL, y_pred_rf):.4f}')
print(f'RF IoU: {compute_iou(y_test_mL, y_pred_rf):.4f}')

In [ ]:
# XGBoost
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb.fit(X_train_mL, y_train_mL)
y_pred_xgb = xgb.predict(X_test_mL)

print(f'XGBoost Accuracy: {accuracy_score(y_test_mL, y_pred_xgb):.4f}')
print(f'XGBoost IoU: {compute_iou(y_test_mL, y_pred_xgb):.4f}')

## 7. DeepLabV3+ Training

In [ ]:
def compute_iou_dl(preds, targets, threshold=0.5):
    preds = (preds > threshold).float()
    intersection = (preds * targets).sum()
    union = preds.sum() + targets.sum() - intersection
    return ((intersection + 1e-7) / (union + 1e-7)).item()

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    total_iou = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            total_loss += criterion(preds, y_batch).item()
            total_iou += compute_iou_dl(preds, y_batch)
    return total_loss / len(loader), total_iou / len(loader)

In [ ]:
model = smp.DeepLabV3Plus(
    encoder_name='efficientnet-b0',
    encoder_weights='imagenet',
    in_channels=N_CHANNELS,
    classes=1,
).to(device)

criterion = smp.losses.FocalLoss(mode='binary', alpha=0.75, gamma=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_iou = 0

print('Training DeepLabV3+...')
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_iou = validate(model, val_loader, criterion)

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), 'best_model_l8.pt')

    print(f'Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | IoU: {val_iou:.4f}')

print(f'\nBest Val IoU: {best_val_iou:.4f}')

## 8. DL Test Evaluation

In [ ]:
model.load_state_dict(torch.load('best_model_l8.pt'))
model.eval()

all_preds = []
all_labels = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch)
        preds_binary = (preds > 0.5).float().cpu().numpy()
        all_preds.append(preds_binary)
        all_labels.append(y_batch.numpy())

dl_preds = np.concatenate(all_preds).flatten()
dl_labels = np.concatenate(all_labels).flatten()

dl_acc = (dl_preds == dl_labels).mean()
dl_iou = compute_iou(dl_labels, dl_preds)

print(f'DeepLabV3+ Accuracy: {dl_acc:.4f}')
print(f'DeepLabV3+ IoU: {dl_iou:.4f}')

## 9. Final Comparison

In [ ]:
print('=' * 50)
print('LANDSAT-8 2017 — CORINGA MANGROVE CLASSIFICATION')
print('=' * 50)
print(f'RF+CT       | Accuracy: {accuracy_score(y_test_mL, y_pred_rf):.4f} | IoU: {compute_iou(y_test_mL, y_pred_rf):.4f}')
print(f'XGBoost     | Accuracy: {accuracy_score(y_test_mL, y_pred_xgb):.4f} | IoU: {compute_iou(y_test_mL, y_pred_xgb):.4f}')
print(f'DeepLabV3+  | Accuracy: {dl_acc:.4f} | IoU: {dl_iou:.4f}')
print('=' * 50)